# HydroSwift Python API Guide & Examples ⚡

HydroSwift automates retrieval, processing, and tidy standardization of Indian hydrological observations across:
- **NWDP** (National Water Data Portal - CKAN DataStore)
- **CWC FFS** (Central Water Commission Flood Forecasting System: River stage & rating-curve discharge)
- **CWC RSMS** (Reservoir Storage Monitoring System: 180+ major reservoirs)
- **India-WRIS REST API** (Official OpenAPI catalog)

---

## Table of Contents
1. [Setup & Version Check](#1-setup--version-check)
2. [The Core Workflow: Find -> Review -> Get -> Use](#2-the-core-workflow-find---review---get---use)
3. [Find: Intelligent Discovery (`hs.search`)](#3-find-intelligent-discovery)
4. [Review: Source Status & Record Estimates (`hs.estimate`)](#4-review-source-status--record-estimates)
5. [Get: Resumable Downloads (`hs.fetch`)](#5-get-resumable-downloads)
6. [Use: Universal Tidy Long DataFrame (`hs.load`)](#6-use-universal-tidy-long-dataframe)
7. [Export & Merging (`hs.merge`)](#7-export--merging)
8. [Automated Plotting (`hs.plot`)](#8-automated-plotting)
9. [CWC Flood Forecasting System & Rating Curves](#9-cwc-flood-forecasting-system--rating-curves)
10. [CWC RSMS Reservoirs (Daily Snapshots & Bulletins)](#10-cwc-rsms-reservoirs)
11. [India-WRIS REST API Client](#11-india-wris-rest-api-client)
12. [Experimental: SWOT Satellite Reservoir Harmonization](#12-experimental-swot-satellite-harmonization)


## 1. Setup & Version Check
Import HydroSwift and check package metadata.

In [ ]:
import hydroswift as hs
import pandas as pd

print(f"Loaded {hs.APP_NAME} v{hs.VERSION_FULL}")
hs.help()

## 2. The Core Workflow: Find -> Review -> Get -> Use
HydroSwift unifies all hydrological operations into four standard steps:
- **Find**: Discover available station/dataset records using `hs.search(...)`
- **Review**: Inspect source health (`cat.attrs['source_status']`) and data volume (`hs.estimate(cat)`)
- **Get**: Download requested items into an output directory with audit logging (`hs.fetch(...)`)
- **Use**: Transform raw observations into one standardized long table (`hs.load(...)`)

## 3. Find: Intelligent Discovery
Search datasets across sources with flexible variable and location queries.

In [ ]:
# Search for rainfall datasets in Karnataka covering 2020 to 2022
cat_rain = hs.search("rainfall", location="Karnataka", period=("2020", "2022"))
cat_rain

In [ ]:
# Exact filter selection: Basin or State
cat_cwc = hs.search("discharge", basin="Krishna", sources=["CWC"])
cat_cwc.head()

## 4. Review: Source Status & Record Estimates
Inspect what each data provider reported and estimate record volumes before pulling data.

In [ ]:
# Per-source status audit (ok, error, rows reported, dropped resources)
for status in cat_rain.attrs.get("source_status", []):
    print(f"{status['source']}: ok={status['ok']}, rows={status['rows']}, note={status['note']}")

In [ ]:
# Server-reported record estimates (exact for NWDP, 'unknown' for CWC)
est = hs.estimate(cat_rain.head(3))
est[["Title", "EstRecords", "EstNote"]]

## 5. Get: Resumable Downloads
Download rows of a catalog into an output directory. Every fetch automatically writes a `manifest.json` detailing endpoints, time spans, and record counts.

In [ ]:
# Download the first two catalog items
run = hs.fetch(cat_rain.head(2), output_dir="output", period=("2020", "2022"))
run

## 6. Use: Universal Tidy Long DataFrame
Transform the downloaded data into HydroSwift's universal tidy format:
`time`, `value`, `variable`, `unit`, `station_id`, `station_name`, `lat`, `lon`, `source`, `method`, `flag`, `file`.

In [ ]:
tidy_df = hs.load(run)
print(f"Loaded {len(tidy_df):,} standardized records.")
tidy_df.head(10)

## 7. Export & Merging
Consolidate multiple downloaded files into a single unified Parquet, GeoPackage, or CSV file.

In [ ]:
# Export to Parquet
hs.merge(run, output="output/karnataka_rainfall_2020_2022.parquet")
print("Exported to output/karnataka_rainfall_2020_2022.parquet")

## 8. Feature-Rich Hydrological Plotting

HydroSwift includes specialized hydrological visualization engines tailored for engineering analysis and water science:
* `kind="timeseries"`: Clean hydrographs with CWC flood alert thresholds (Warning, Danger, HFL lines).
* `kind="hydrograph"`: Dual-axis river stage & discharge hydrographs.
* `kind="hyetograph"`: Precipitation hyetograph bars with cumulative rainfall mass curves.
* `kind="rating_curve"`: Empirical stage-discharge ($H-Q$) hydraulic rating curves with fitted equations and $R^2$.
* `kind="compare"`: Multi-station comparison on synchronized time axes.
* `kind="fdc"`: Flow duration curves with $Q_{10}$, $Q_{50}$, and $Q_{90}$ exceedance percentiles.
* `interactive=True`: Interactive Plotly HTML figures with hover tooltips and range sliders.

In [ ]:
# 1. Standard time series hydrograph with flood alert thresholds
ts_plots = hs.plot(run, output_dir="output/plots", kind="timeseries")
print(f"Generated {len(ts_plots)} time-series plots in output/plots/")

# 2. Dual-axis hydrograph (stage on primary Y, discharge on secondary Y)
# hs.plot(tidy_df, kind="hydrograph")

# 3. Empirical hydraulic rating curve (H-Q scatter + fitted curve)
# hs.plot(tidy_df, kind="rating_curve")

# 4. Precipitation hyetograph (intensity bars + cumulative curve)
# hs.plot(tidy_df, kind="hyetograph")

## 9. Interactive Geospatial Mapping & Map Layers

Generate self-contained, publication-ready interactive Leaflet maps directly from search catalogs or tidy DataFrames:
* **Basemap layer switcher**: OpenStreetMap, CartoDB Light, CartoDB Dark, Esri Satellite.
* **Marker clustering**: Smooth, high-performance rendering of hundreds of stations.
* **Flood alert highlights**: Pulsing indicators for stations exceeding Warning or Danger levels.
* **Rich popups**: Station codes, river basin, state, latest telemetry readings, and thresholds.
* **Custom GeoJSON layers**: Overlay watershed polygons, catchment boundaries, and river networks.

In [ ]:
# Create interactive map from discovered stations
hmap = hs.map(cat_rain, output="output/karnataka_rainfall_map.html", basemap="carto-light")
print(f"Created interactive map with {hmap.stations_count} stations: {hmap.path}")
# In Jupyter, calling `hmap` directly renders the interactive map inline!
hmap

## 10. CWC Flood Forecasting System & Rating Curves
CWC operational river telemetry tracks sub-daily river stage. HydroSwift derives discharge using hydraulic rating curves (GUARDIAN 2024 methodology).

In [ ]:
# Discover CWC river gauging stations for the Godavari basin
cwc_godavari = hs.search("discharge", basin="Godavari", sources=["CWC"])
cwc_godavari.head(5)

## 11. CWC RSMS Reservoirs
Retrieve live storage, capacities, and weekly bulletin archives for 180+ major national reservoirs.

In [ ]:
# Fetch latest daily storage snapshot
res_daily = hs.rsms.daily(output_path="output/cwc_rsms/daily_snapshot.csv")
res_daily.head()

In [ ]:
# Discover published weekly bulletins
bulletins = hs.rsms.bulletins(from_date="2025-01-01", to_date="2025-06-01")
bulletins.head()

## 12. India-WRIS REST API Client
Query granular station-level time series via the official India-WRIS OpenAPI 3.0 catalog.

In [ ]:
from hydroswift.sources.wris_api import WrisClient

wris = WrisClient()
# Query station observations by State and District
admin_data = wris.fetch_admin(
    variable="rainfall",
    state_name="Andhra Pradesh",
    district_name="Kadapa",
    start_date="2024-12-01",
    end_date="2024-12-05",
    agency_name="CWC",
    size=5,
)
pd.DataFrame(admin_data)[["stationCode", "stationName", "dataTime", "dataValue", "unit"]].head()

## 13. Experimental: SWOT Satellite Harmonization
Harmonize NASA/CNES SWOT KaRIn water surface elevations (+/- 3.5 days) with in-situ reservoir storage bulletins.

In [ ]:
from hydroswift.experimental.swot import build_reservoir_database

# Staged for advanced satellite remote-sensing validation workflows
print("SWOT reservoir database builder is available in hydroswift.experimental.swot")